In [1]:
%pip install kagglehub

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
#Imports 
import pandas as pd
from pathlib import Path

In [3]:
#path to save the database 

data_path = Path("D:\customer_retention\data\online_retail_II.csv")
df = pd.read_csv(data_path, encoding="latin-1")

print(f"\n Dataset shape: {df.shape}")


 Dataset shape: (1067371, 8)


In [4]:
display(df.head())

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   Invoice      1067371 non-null  object 
 1   StockCode    1067371 non-null  object 
 2   Description  1062989 non-null  object 
 3   Quantity     1067371 non-null  int64  
 4   InvoiceDate  1067371 non-null  object 
 5   Price        1067371 non-null  float64
 6   Customer ID  824364 non-null   float64
 7   Country      1067371 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 65.1+ MB


In [6]:
df.isnull().sum()
df.describe()

,Quantity,Price,Customer ID
count,1.067371e+06,1.067371e+06,824364.000000
mean,9.938898e+00,4.649388e+00,15324.638504
std,1.727058e+02,1.235531e+02,1697.464450
min,-8.099500e+04,-5.359436e+04,12346.000000
25%,1.000000e+00,1.250000e+00,13975.000000
50%,3.000000e+00,2.100000e+00,15255.000000
75%,1.000000e+01,4.150000e+00,16797.000000
max,8.099500e+04,3.897000e+04,18287.000000


Customer ID completeness: The dataset contains 1,067,371 rows, and 824,364 rows have a valid Customer ID. That means 243,007 rows, about 22.8%, are missing Customer IDs. Since customer-level metrics like RFM require customer identity, rows without Customer ID should be excluded from customer analytics and the rationale should be documented.

Quantity quality: The minimum Quantity is -80,995, indicating return/cancellation behavior rather than valid sales demand. Negative quantities should be excluded when calculating revenue or purchase-volume KPIs, while optionally preserved in a separate returns analysis.

Price quality: The minimum Unit Price is -53,594, which suggests adjustments/corrections and non-sale records. Negative prices should be removed for sales and revenue calculations to avoid distorted metrics

In [7]:
df_raw = df.copy()


In [8]:
df_clean = df.copy()
df_clean = df_clean[df_clean["Customer ID"].notna()]
df_clean = df_clean[df_clean["Quantity"] >= 0]
df_clean = df_clean[df_clean["Price"] > 0]


In [9]:
df_clean["Revenue"] = df_clean["Quantity"] * df_clean["Price"]

In [10]:
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])
print(df_clean['InvoiceDate'].dtype)

datetime64[ns]


In [11]:
df_clean['InvoiceDay'] = df_clean['InvoiceDate'].dt.date
df_clean['YearMonth'] = df_clean['InvoiceDate'].dt.to_period('M')
display(df_clean[['InvoiceDate', 'InvoiceDay', 'YearMonth']].head())

,InvoiceDate,InvoiceDay,YearMonth
0,2009-12-01 07:45:00,2009-12-01,2009-12
1,2009-12-01 07:45:00,2009-12-01,2009-12
2,2009-12-01 07:45:00,2009-12-01,2009-12
3,2009-12-01 07:45:00,2009-12-01,2009-12
4,2009-12-01 07:45:00,2009-12-01,2009-12


In [12]:
print("=== Cleaned dataset summary ===")
print(f"Rows before cleaning: {len(df_raw):,}")
print(f"Rows after cleaning: {len(df_clean):,}")
print(f"Rows removed: {len(df_raw) - len(df_clean):,}")
print(f"Total cleaned revenue: {df_clean['Revenue'].sum():,.2f}")

=== Cleaned dataset summary ===
Rows before cleaning: 1,067,371
Rows after cleaning: 805,549
Rows removed: 261,822
Total cleaned revenue: 17,743,429.18
